In [10]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy

today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    print(mode(data))
    return fig

In [2]:
filename = '/home/ctrader/region_similarity/data/9606.protein.links.full.v11.5.txt'
f = open(filename, 'r')
header = f.readline().strip().split(' ')
f.close()
string_genes_arr = np.loadtxt(filename, dtype=str, skiprows=1, usecols=[0, 1])
string_vals_arr = np.loadtxt(filename, dtype=int, skiprows=1, usecols=np.arange(2, len(header)))

In [3]:
val_header = header[2:]

In [4]:
np.sort(val_header)

array(['coexpression', 'coexpression_transferred', 'combined_score',
       'cooccurence', 'database', 'database_transferred', 'experiments',
       'experiments_transferred', 'fusion', 'homology', 'neighborhood',
       'neighborhood_transferred', 'textmining', 'textmining_transferred'],
      dtype='<U24')

In [5]:
neigh = [0, 1]
fusion = [2]
cooccur = [3]
homology = [4]
coex = [5, 6]
exper = [7, 8]
database = [9, 10]
textmin = [11, 12]
combined_score = [13]
metric_inds = [coex, cooccur, database, exper, fusion, homology, neigh, textmin, combined_score]

In [6]:
unique_string_genes = np.unique(string_genes_arr)

In [137]:
#indicies for each gene in alphabetical order
protein2ind = {}
for gene, ind in zip(unique_string_genes, np.arange(len(unique_string_genes))):
    protein2ind[gene] = ind

In [139]:
#gene ids -> indices
string_gene_inds_arr = np.array(list(map(protein2ind.get, string_genes_arr.flatten())), 
                                dtype=int).reshape(string_genes_arr.shape)

In [12]:
#for each metric, populate [metric, protein1, protein2] array
metric_protein1_protein2 = []
for col_inds in metric_inds:
    print(np.array(val_header)[col_inds])
    #combining human and transferred scores if applicable
    val_arr = np.max(string_vals_arr[:, col_inds], axis=1)
    metric_sparse = sp.sparse.coo_matrix((val_arr, 
                                          (string_gene_inds_arr[:, 0], string_gene_inds_arr[:, 1])))
    metric_protein1_protein2.append(metric_sparse.toarray())
metric_protein1_protein2 = np.array(metric_protein1_protein2)

['coexpression' 'coexpression_transferred']
['cooccurence']
['database' 'database_transferred']
['experiments' 'experiments_transferred']
['fusion']
['homology']
['neighborhood' 'neighborhood_transferred']
['textmining' 'textmining_transferred']
['combined_score']


In [27]:
metric_protein1_protein2.shape

(9, 19385, 19385)

In [17]:
metric_order = ['coexpression', 'cooccurence', 'database', 'experiments', 
                'fusion', 'homology', 'neighborhood', 'textmining', 'combined_score']

In [36]:
outpath = '/home/ctrader/region_similarity/data/rank_based/9606.protein.links.full.v11.5.allMetrics.proteinXProteinSTRINGScore.hdf5'
out_f = h5py.File(outpath, 'w')
dset = out_f.create_dataset('metric_protein1_protein2', data=metric_protein1_protein2)
dset = out_f.create_dataset('metric_order', data=np.char.encode(metric_order))
dset = out_f.create_dataset('protein_order', data=np.char.encode(unique_string_genes))
out_f.close()

In [78]:
unique_string_genes

array(['9606.ENSP00000000233', '9606.ENSP00000000412',
       '9606.ENSP00000001008', ..., '9606.ENSP00000485668',
       '9606.ENSP00000485672', '9606.ENSP00000485678'], dtype='<U20')

In [ ]:
#for each region, find the number of interconnections above some threshold

In [83]:
#load mapped gene ids
gene_id_mapped_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221027.proteinidmapped.bed'
f = open(gene_id_mapped_path, 'r')
all_regions_gene_ids = []
unique_string_genes_set = set(unique_string_genes)
for line in f.readlines():
    region_gene_ids = []
    geneids = re.split('\t|;|\|', line.strip())[3:]
    for geneid in geneids:
        geneid = '9606.' + geneid.split('.')[0] #removing everything after ., adding 9606.
        if geneid in unique_string_genes_set:
            region_gene_ids.append(geneid)
    all_regions_gene_ids.append(list(np.unique(region_gene_ids)))
f.close()

In [283]:
def rank_regions(arr):
    #rank each region
    ranked_regions = rankdata(arr, axis=0)

    #find the number of regions with 0 values for each component
    comp_num_zero = len(arr) - np.count_nonzero(arr, axis=0)

    #adjusting/resetting ranks
    ranked_regions = ranked_regions - comp_num_zero
    ranked_regions = np.where(ranked_regions < 0, 0, ranked_regions)

    return ranked_regions

In [290]:
for thresh in [0, 100, 200, 300, 400, 500, 600, 700, 800, 900]:
    print(thresh)
    all_regions_counts_per_metric = []
    for region in range(len(all_regions_gene_ids)):
        top_protein_ind = ''
        top_protein_ids_meeting_thresh = np.array([])
        #finding top gene for region
        for protein_id in all_regions_gene_ids[region]:
            protein_ind = protein2ind[protein_id]
            combined_scores = metric_protein1_protein2[-1, protein_ind, :]
            protein_ids_meeting_thresh = np.arange(len(combined_scores))[combined_scores > thresh]
            if len(protein_ids_meeting_thresh) > len(top_protein_ids_meeting_thresh): 
                top_protein_ind = protein_ind
                top_protein_ids_meeting_thresh = protein_ids_meeting_thresh
        #for best gene/protein, find counts from each metric
        if top_protein_ind != '':
            top_protein_arr = metric_protein1_protein2[:, top_protein_ind, top_protein_ids_meeting_thresh]
            counts_for_region = np.count_nonzero(top_protein_arr, axis=1)
        else:
            counts_for_region = np.zeros(len(metric_protein1_protein2), dtype=int)
        all_regions_counts_per_metric.append(counts_for_region)
    all_regions_counts_per_metric = np.array(all_regions_counts_per_metric)
    
    #ranking regions
    ranked_regions = rank_regions(all_regions_counts_per_metric)
    
    #max ranks for each individual STRING metric
    max_ranks = np.max(ranked_regions, axis=0)

    outpath = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221115.top_gene.combined_scores_above_' + str(thresh) + '.counts_by_metric.hdf5'
    out_f = h5py.File(outpath, 'w')
    dset = out_f.create_dataset('all_regions_counts_per_metric', data=all_regions_counts_per_metric)
    dset = out_f.create_dataset('ranked_regions', data=ranked_regions)
    dset = out_f.create_dataset('max_ranks', data=max_ranks)
    dset = out_f.create_dataset('comp_order', data=np.char.encode(metric_order))
    out_f.close()

0
100
200
300
400
500
600
700
800
900


In [293]:
np.max(ranked_regions, axis=0)

array([32844.5,  2831. , 29478.5, 32906.5,  1315.5, 11757.5,  4002. ,
       37032.5, 37216.5])

In [291]:
max_ranks

array([32844.5,  2831. , 29478.5, 32906.5,  1315.5, 11757.5,  4002. ,
       37032.5, 37216.5])

In [289]:
rank_regions(all_regions_counts_per_metric)

array([[    0. ,     0. ,     0. , ...,     0. ,     0. ,     0. ],
       [    0. ,     0. ,     0. , ...,     0. ,     0. ,     0. ],
       [    0. ,     0. ,     0. , ...,     0. ,     0. ,     0. ],
       ...,
       [27837. ,     0. , 22719. , ...,     0. , 29961. , 29665. ],
       [27837. ,     0. , 22719. , ...,     0. , 29961. , 29665. ],
       [ 7148. ,     0. ,  3950.5, ...,     0. ,  6765.5,  6784.5]])

**Gencode file with expanded regulatory domains (as in GREAT)**

In [4]:
f = open('/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.gtf', 'r')
gencode_file = f.readlines()
f.close()

In [5]:
data = []
for line in gencode_file[5:]:
    line = re.split('\t|;| ', line.strip())
    #gene coordinates
    gene_chrm = line[0]
    gene_start = line[3]
    gene_end = line[4]
    gene_strand = line[6]
    gene_type = line[2]
    
    #gene name
    gene_name_index = line.index('gene_name')
    gene_name = line[gene_name_index + 1].strip("\"")
    
    #gene id (ENSG)
    gene_id_index = line.index('gene_id')
    gene_id = line[gene_id_index + 1].strip("\"")
    #protein id (ENSP)
    if 'protein_id' in line:
        protein_id_index = line.index('protein_id')
        protein_id = line[protein_id_index + 1].strip("\"")
    else:
        protein_id = ''
    #promoter start and end
    if gene_strand == '+':
        promoter_start = gene_start
        promoter_end = str(int(gene_start) + 1)
        basal_start = str(int(gene_start) - 5000)
        basal_end = str(int(gene_start) + 1000)
        domain_start = str(int(gene_start) - 1000000)
        domain_end = str(int(gene_end) + 1000000)
        
    elif gene_strand == '-':
        promoter_start = str(int(gene_end) - 1)
        promoter_end = gene_end
        basal_start = str(int(gene_end) - 1000)
        basal_end = str(int(gene_end) + 5000)
        domain_start = str(int(gene_start) - 1000000)
        domain_end = str(int(gene_end) + 1000000)
        
    data_row = [gene_chrm, gene_start, gene_end, gene_name, gene_type, gene_strand, 
                promoter_start, promoter_end, basal_start, basal_end, domain_start, 
                domain_end, gene_id, protein_id]
    data.append(data_row)

data = np.array(data)

header = ['chrm', 'start', 'end', 'name', 'type', 'strand', 'Promoter start (bp)', 
          'Promoter end (bp)', 'Basal start (bp)', 'Basal end (bp)', 'Domain start (bp)', 
          'Domain end (bp)', 'gene id', 'protein id']

In [6]:
data.shape

(1968859, 14)

In [7]:
#limiting domain to real chromosome positions
chrm = data[:, 0]
tss = data[:, 6].astype(int)
domain_m = data[:, 8:12].astype(int)
names = data[:, 3]

domain_m = np.where(domain_m < 1, 1, domain_m)

In [8]:
domain_m

array([[   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1012227],
       ...,
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023]])

In [9]:
#creating chromosome masked data
chrm_dom_dict = {}
chrm_tss_dict = {}
chrm_name_dict = {}

for chromosome in np.unique(chrm):
    chrm_mask = chrm==chromosome
    chrm_doms = domain_m[chrm_mask, :]
    chrm_tss = tss[chrm_mask]
    chrm_names = names[chrm_mask]
    
    chrm_dom_dict[chromosome] = chrm_doms
    chrm_tss_dict[chromosome] = chrm_tss
    chrm_name_dict[chromosome] = chrm_names

In [43]:
chrm_dom_dict['chr1'].shape

(191060, 4)

In [8]:
chrm_name_dict['chr2']

array(['FAM110C', 'FAM110C', 'FAM110C', ..., 'RPL23AP88', 'RPL23AP88',
       'RPL23AP88'], dtype='<U25')

In [10]:
domain_m

array([[   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1012227],
       ...,
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023]])

In [14]:
gene_dom.shape

(4,)

In [25]:
gene

5655

In [47]:
for gene in range(len(domain_m))[191058:191062]:
    if gene%50000==0:
        print(gene)
    
    gene_chrm = chrm[gene]
    gene_tss = tss[gene]
    gene_name = names[gene]
    gene_dom = domain_m[gene, :]
    
    #filtering out wrong chromosomes
#     chrm_mask = chrm==gene_chrm
#     chrm_doms = domain_m[chrm_mask, :]
#     chrm_tss = tss[chrm_mask]
#     chrm_names = names[chrm_mask]
    
    chrm_doms = chrm_dom_dict[gene_chrm]
    chrm_tss = chrm_tss_dict[gene_chrm]
    chrm_names = chrm_name_dict[gene_chrm]
    print(gene_chrm)
    print(chrm_doms.shape)
    print()
    
    #filtering out other annotations of the same gene
    name_mask = chrm_names!=gene_name
    chrm_name_doms = chrm_doms[name_mask, :]
    chrm_name_tss = chrm_tss[name_mask]
    
    #find all the genes to the left and right
    left_genes = chrm_name_tss<gene_tss
    right_genes = chrm_name_tss>gene_tss
    left_doms = chrm_name_doms[left_genes, :]
    right_doms = chrm_name_doms[right_genes, :]

    #if there are any genes, find the closest basal region
    if len(left_doms) > 0:
        max_left_basal_end = np.max(left_doms[:, 1])
    else:
        max_left_basal_end = gene_dom[2]
    if len(right_doms) > 0:
        min_right_basal_start = np.min(right_doms[:, 0])
    else:
        min_right_basal_start = gene_dom[3]

    #setting the domain for each gene
    gene_dom[2] = np.min([np.max([max_left_basal_end, gene_dom[2]]), gene_dom[0]])
    gene_dom[3] = np.max([np.min([min_right_basal_start, gene_dom[3]]), gene_dom[1]])

chr1
(191060, 4)

chr1
(191060, 4)

chr2
(147486, 4)

chr2
(147486, 4)



In [18]:
domain_m[:gene]

array([[   6869,   12869,       1,   13501],
       [   6869,   12869,       1,   13501],
       [   6869,   12869,       1,   13501],
       ...,
       [2493479, 2499479, 2493479, 2499479],
       [2492853, 2498853, 2492853, 2498853],
       [2525597, 2531597, 2506211, 2531597]])

In [ ]:
#changing domain values
data[:, 8:12] = domain_m.astype(str)

In [15]:
'\t'.join(header)

'chrm\tstart\tend\tname\ttype\tstrand\tPromoter start (bp)\tPromoter end (bp)\tBasal start (bp)\tBasal end (bp)\tDomain start (bp)\tDomain end (bp)\tgene id\tprotein id'

In [ ]:
# outfile = '/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.parsed.promoterbp.extendeddomain.CT20221026.txt'
# np.savetxt(outfile, data, delimiter='\t', fmt='%s', header='\t'.join(header))

In [31]:
f = open('/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.parsed.promoterbp.extendeddomain.CT20221026.txt', 'r')
new_gencode_file = f.readlines()
f.close()

In [15]:
new_gencode_file[1]

'chr1\t11869\t14409\tDDX11L1\tgene\t+\t11869\t11870\t6869\t12869\t1\t13501\tENSG00000223972.5\tNA\n'

In [16]:
line.split('\t')

AttributeError: 'list' object has no attribute 'split'

In [32]:
new_data = []
for line in new_gencode_file[1:]:
    new_data.append(line.strip().split('\t'))
new_data = np.array(new_data)

In [49]:
np.unique(new_data[:, 4])

array(['CDS', 'Selenocysteine', 'UTR', 'exon', 'gene', 'start_codon',
       'stop_codon', 'transcript'], dtype='<U25')

In [33]:
np.sum(new_data[:, 8:12][:gene].astype(int) - domain_m[:gene])

0